## **Bibliotecas**

In [4]:
#%pip install pydantic[email]

from datetime import datetime
from typing import Optional
from uuid import uuid4

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, EmailStr, Field, field_serializer, UUID4

## **FastAPI**

In [5]:
app = FastAPI()

## **Classes**

In [6]:
class User(BaseModel):

    # forbid: Garante que a API rejeite qualquer campo enviado que não esteja definido aqui.
    model_config = {
        "extra": "forbid",
    }

    # Simulação de banco de dados em memória    
    __users__ = []

    name: str = Field(
        ...,
        description="Name of the user"
    )

    email: EmailStr = Field(
        ...,
        description="Email address of the user"
    )

    # list[UUID4]: Define uma lista de IDs únicos.
    friends: list[UUID4] = Field(
        default_factory=list,
        max_length=500,  # Limita o tamanho da lista
        description="List of friends"
    )

    blocked: list[UUID4] = Field(
        default_factory=list,
        max_length=500,
        description="List of blocked users"
    )

    # Optional[datetime]: O campo pode ser nulo ou conter uma data e hora.
    signup_ts: Optional[datetime] = Field(
        default_factory=datetime.now,  # Gera automaticamente a data/hora do momento do cadastro.
        description="Signup timestamp"
    )

    id: UUID4 = Field(
        default_factory=uuid4,
        description="Unique identifier"
    )


    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        """
        Converte o objeto UUID em uma string simples 
        quando os dados são enviados via JSON.
        """
        
        return str(id)

## **Endpoints**

In [7]:
# @app.get: Define uma rota de leitura.
# response_model: Indica que a saída será uma lista de objetos do tipo User.
@app.get("/users", response_model=list[User])
async def get_users() -> list[User]:
    """
    Retorna a lista de todos os usuários cadastrados.
    """

    return list(User.__users__)

In [8]:
# @app.post: Define uma rota de criação.
@app.post("/users", response_model=User)
async def create_user(user: User):
    """
    Cria um novo usuário no sistema e realiza a validação automática de dados.
    """

    User.__users__.append(user)
    return user

In [9]:
# @app.get("/{user_id}"): Define uma rota dinâmica onde a parte entre chaves é uma variável.
@app.get("/users/{user_id}", response_model=User)
async def get_user(user_id: UUID4) -> User | JSONResponse:
    """
    Busca um usuário pelo seu ID.
    """

    try:
        # next(): Tenta encontrar o primeiro usuário na lista __users__ que 
        # possua o ID igual ao user_id passado na URL da API.
        return next((user for user in User.__users__ if user.id == user_id))
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "User not found"})

## **Main**

In [10]:
with TestClient(app) as client:
    for i in range(5):
        response = client.post(
            "/users",
            json={
                "name": f"User {i}",
                "email": f"example{i}@arjancodes.com"
            },
        )
        assert response.status_code == 200
        assert response.json()["name"] == f"User {i}", (
            "The name of the user should be User {i}"
        )
        assert response.json()["id"], "The user should have an id"

        user = User.model_validate(response.json())
        assert str(user.id) == response.json()["id"], "The id should be the same"
        assert user.signup_ts, "The signup timestamp should be set"
        assert user.friends == [], "The friends list should be empty"
        assert user.blocked == [], "The blocked list should be empty"

    response = client.get("/users")
    assert response.status_code == 200, "Response code should be 200"
    assert len(response.json()) == 5, "There should be 5 users"

    response = client.post(
        "/users",
        json={
            "name": "User 5",
            "email": "example5@arjancodes.com"
        }
    )
    assert response.status_code == 200
    assert response.json()["name"] == "User 5", (
        "The name of the user should be User 5"
    )
    assert response.json()["id"], "The user should have an id"

    user = User.model_validate(response.json())
    assert str(user.id) == response.json()["id"], "The id should be the same"
    assert user.signup_ts, "The signup timestamp should be set"
    assert user.friends == [], "The friends list should be empty"
    assert user.blocked == [], "The blocked list should be empty"

    response = client.get(f"/users/{response.json()['id']}")
    assert response.status_code == 200
    assert response.json()["name"] == "User 5", (
        "This should be the newly created user"
    )

    response = client.get(f"/users/{uuid4()}")
    assert response.status_code == 404
    assert response.json()["message"] == "User not found", (
        "We technically should not find this user"
    )

    response = client.post("/users", json={"name": "User 6", "email": "wrong"})
    assert response.status_code == 422, "The email address is should be invalid"